# Machine Learning

In [224]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import warnings
from scipy.stats import uniform, randint
import sys
import timm
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, random_split, DataLoader, Subset, ConcatDataset
import copy

In [225]:
initial_df = pd.read_parquet("../data/processed/anime_data_2.parquet")
initial_df.info()

<class 'pandas.DataFrame'>
Index: 5350 entries, 0 to 71
Data columns (total 84 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   mal_id                  5350 non-null   int64  
 1   producers               5350 non-null   object 
 2   genres                  5350 non-null   object 
 3   studios                 5350 non-null   object 
 4   demographics            5350 non-null   object 
 5   themes                  5350 non-null   object 
 6   rating                  5309 non-null   str    
 7   members                 5350 non-null   int64  
 8   thumbnail               5350 non-null   bool   
 9   prequel_score           5350 non-null   float64
 10  prequel_members         5350 non-null   float64
 11  prequel_type            5350 non-null   str    
 12  cohort                  5350 non-null   str    
 13  wc_z                    5350 non-null   float64
 14  forum_z                 5350 non-null   float64
 15  favor

In [226]:
df = initial_df.head(5278)
real_df = initial_df.tail(72)

print(df.info())
print(real_df.info())

<class 'pandas.DataFrame'>
Index: 5278 entries, 0 to 5307
Data columns (total 84 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   mal_id                  5278 non-null   int64  
 1   producers               5278 non-null   object 
 2   genres                  5278 non-null   object 
 3   studios                 5278 non-null   object 
 4   demographics            5278 non-null   object 
 5   themes                  5278 non-null   object 
 6   rating                  5278 non-null   str    
 7   members                 5278 non-null   int64  
 8   thumbnail               5278 non-null   bool   
 9   prequel_score           5278 non-null   float64
 10  prequel_members         5278 non-null   float64
 11  prequel_type            5278 non-null   str    
 12  cohort                  5278 non-null   str    
 13  wc_z                    5278 non-null   float64
 14  forum_z                 5278 non-null   float64
 15  fav

## Input Preparation

Right now, the priority is to reduce dimensions. The plan is the following:
* Reduce the dimensions of the image tensors from 512
* Reduce the dimensions of sentimental analysis tensors from 768
* Reduce the pool of producers and studios into embedded vectors

In [227]:
df = df.reset_index(drop=True)

all_indices = np.arange(len(df))

train_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.10,
    random_state=42
)

train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.10,
    random_state=42
)

print(df.index[:5])
print(train_idx[:5])

RangeIndex(start=0, stop=5, step=1)
[2045 3165  518 1851  622]


In [228]:
studio_to_idx = {"<UNK>": 0}
producer_to_idx = {"<UNK>": 0}

for studios in df.iloc[train_idx]["studios"]:
    for studio in studios:
        if studio not in studio_to_idx:
            studio_to_idx[studio] = len(studio_to_idx)

for producers in df.iloc[train_idx]["producers"]:
    for producer in producers:
        if producer not in producer_to_idx:
            producer_to_idx[producer] = len(producer_to_idx)

n_studios = len(studio_to_idx)
n_producers = len(producer_to_idx)

print("Number of studios:", n_studios)
print("Number of producers:", n_producers)

Number of studios: 483
Number of producers: 1003


In [229]:
def get_studio_indices(studios):
    return [
        studio_to_idx.get(studio, 0)
        for studio in studios
    ]

def get_producer_indices(producers):
    return [
        producer_to_idx.get(producer, 0)
        for producer in producers
    ]

df["studio_idx"] = df["studios"].apply(get_studio_indices)
df["producer_idx"] = df["producers"].apply(get_producer_indices)

real_df["studio_idx"] = real_df["studios"].apply(get_studio_indices)
real_df["producer_idx"] = real_df["producers"].apply(get_producer_indices)

print(df["studio_idx"].head())
print(df["producer_idx"].head())

0     [8]
1    [14]
2     [8]
3    [19]
4    [87]
Name: studio_idx, dtype: object
0       [7, 238, 263]
1          [238, 239]
2        [7, 41, 238]
3             [2, 41]
4    [2, 137, 13, 33]
Name: producer_idx, dtype: object


In [230]:
def create_embedding_bag_inputs(index_lists):
    flat_indices = []
    offsets = []

    current_offset = 0

    for indices in index_lists:
        offsets.append(current_offset)
        flat_indices.extend(indices)
        current_offset += len(indices)

    return (
        torch.tensor(flat_indices, dtype=torch.long),
        torch.tensor(offsets, dtype=torch.long)
    )


studio_indices, studio_offsets = create_embedding_bag_inputs(
    df["studio_idx"]
)

producer_indices, producer_offsets = create_embedding_bag_inputs(
    df["producer_idx"]
)

real_studio_indices, real_studio_offsets = create_embedding_bag_inputs(
    real_df["studio_idx"]
)

real_producer_indices, real_producer_offsets = create_embedding_bag_inputs(
    real_df["producer_idx"]
)

print("Studio indices:", studio_indices.shape)
print("Studio offsets:", studio_offsets.shape)

print("Producer indices:", producer_indices.shape)
print("Producer offsets:", producer_offsets.shape)

Studio indices: torch.Size([5656])
Studio offsets: torch.Size([5278])
Producer indices: torch.Size([17800])
Producer offsets: torch.Size([5278])


In [231]:
def split_embedding_bag_inputs(index_lists, train_idx, val_idx, test_idx):
    
    def create_for_rows(rows):
        selected_lists = [index_lists[i] for i in rows]
        return create_embedding_bag_inputs(selected_lists)

    train_indices, train_offsets = create_for_rows(train_idx)
    val_indices, val_offsets = create_for_rows(val_idx)
    test_indices, test_offsets = create_for_rows(test_idx)

    return (
        train_indices, train_offsets,
        val_indices, val_offsets,
        test_indices, test_offsets
    )


(
    studio_indices_train,
    studio_offsets_train,
    studio_indices_val,
    studio_offsets_val,
    studio_indices_test,
    studio_offsets_test
) = split_embedding_bag_inputs(
    df["studio_idx"].tolist(),
    train_idx,
    val_idx,
    test_idx
)


(
    producer_indices_train,
    producer_offsets_train,
    producer_indices_val,
    producer_offsets_val,
    producer_indices_test,
    producer_offsets_test
) = split_embedding_bag_inputs(
    df["producer_idx"].tolist(),
    train_idx,
    val_idx,
    test_idx
)

### Synopsis

In [232]:
initial_semantic_embeddings = np.load('../data/processed/semantic_embeddings.npy')
print(initial_semantic_embeddings.shape)
semantic_embeddings = initial_semantic_embeddings[:5278]
real_semantic_embeddings = initial_semantic_embeddings[-72:]

(5350, 768)


In [233]:
text_scaler = StandardScaler()

text_scaler.fit(
    semantic_embeddings[train_idx]
)

semantic_embeddings_train = text_scaler.transform(
    semantic_embeddings[train_idx]
)

semantic_embeddings_val = text_scaler.transform(
    semantic_embeddings[val_idx]
)

semantic_embeddings_test = text_scaler.transform(
    semantic_embeddings[test_idx]
)

real_semantic_embeddings = text_scaler.transform(
    real_semantic_embeddings
)

print(
    "Train text NaNs:",
    np.isnan(semantic_embeddings_train).sum()
)

print(
    "Val text NaNs:",
    np.isnan(semantic_embeddings_val).sum()
)

print(
    "Test text NaNs:",
    np.isnan(semantic_embeddings_test).sum()
)

print(
    "Train text infs:",
    np.isinf(semantic_embeddings_train).sum()
)

Train text NaNs: 0
Val text NaNs: 0
Test text NaNs: 0
Train text infs: 0


In [234]:
class LearnedProjector(nn.Module):
    """
    Projects a frozen all-mpnet-base-v2 sentence embedding (768-dim)
    down to a smaller learned representation via a linear layer.

    Note: sentence-transformers' mpnet output is L2-normalized by default
    (normalize_embeddings=True), so no extra normalization is applied here
    on the input side.

    Usage:
        projector = LearnedProjector(in_dim=768, out_dim=64)
        z = projector(x)  # x: (batch, 768) -> z: (batch, 64)
    """
    def __init__(self, in_dim: int = 768, out_dim: int = 64,
                 hidden_dim: int | None = None, dropout: float = 0.1):
        super().__init__()

        if hidden_dim is None:
            self.net = nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.LayerNorm(out_dim)
            )
        else:
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, out_dim),
                nn.LayerNorm(out_dim)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

projector = LearnedProjector(in_dim=768, out_dim=64)

## Images

In [235]:
image_data = np.load("../data/processed/image_embeddings.npy")

In [236]:
image_scaler = StandardScaler()

image_scaler.fit(
    image_data[train_idx]
)

image_train = image_scaler.transform(
    image_data[train_idx]
)

image_val = image_scaler.transform(
    image_data[val_idx]
)

image_test = image_scaler.transform(
    image_data[test_idx]
)

image_real = image_scaler.transform(
    image_data[-72:]
)

print("Image train NaNs:", np.isnan(image_train).sum())
print("Image val NaNs:", np.isnan(image_val).sum())
print("Image test NaNs:", np.isnan(image_test).sum())

Image train NaNs: 0
Image val NaNs: 0
Image test NaNs: 0


## Fusion Network

### Score Prediction

Before making the network, let's confirm dimensions in the "other" category since it's unclear.

In [237]:
print(df.info())
print(real_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 5278 entries, 0 to 5277
Data columns (total 86 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   mal_id                  5278 non-null   int64  
 1   producers               5278 non-null   object 
 2   genres                  5278 non-null   object 
 3   studios                 5278 non-null   object 
 4   demographics            5278 non-null   object 
 5   themes                  5278 non-null   object 
 6   rating                  5278 non-null   str    
 7   members                 5278 non-null   int64  
 8   thumbnail               5278 non-null   bool   
 9   prequel_score           5278 non-null   float64
 10  prequel_members         5278 non-null   float64
 11  prequel_type            5278 non-null   str    
 12  cohort                  5278 non-null   str    
 13  wc_z                    5278 non-null   float64
 14  forum_z                 5278 non-null   float64
 15

In [238]:
X_other_pre = df.drop(columns=['members', 'studio_idx', 'producer_idx', 'mal_id', 'cohort', 'producers', 'genres', 'studios', 'demographics', 'themes', 'score_z', 'wc_z', 'favorites_z', 'drop_rate_z', 'forum_z'])
X_other_pre = X_other_pre.reset_index(drop=True)
X_other_pre = pd.get_dummies(X_other_pre, columns=['rating', 'prequel_type'], dtype=int)
X_other_pre.columns = X_other_pre.columns.str.replace(' ', '_')
print(X_other_pre.info())

real_other_pre = real_df.drop(columns=['members', 'studio_idx', 'producer_idx', 'mal_id', 'cohort', 'producers', 'genres', 'studios', 'demographics', 'themes', 'score_z', 'wc_z', 'favorites_z', 'drop_rate_z', 'forum_z'])
real_other_pre = real_other_pre.reset_index(drop=True)
real_other_pre = pd.get_dummies(real_other_pre, columns=['rating', 'prequel_type'], dtype=int)
real_other_pre[[f'prequel_type_{type}' for type in ['Movie', 'Music', 'ONA', 'OVA', 'PV', 'Special', 'TV_Special']]] = 0
real_other_pre['rating_R+_-_Mild_Nudity'] = 0
real_other_pre.columns = real_other_pre.columns.str.replace(' ', '_')
print(real_other_pre.info())

<class 'pandas.DataFrame'>
RangeIndex: 5278 entries, 0 to 5277
Data columns (total 83 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   thumbnail                              5278 non-null   bool   
 1   prequel_score                          5278 non-null   float64
 2   prequel_members                        5278 non-null   float64
 3   adaptation_score                       5278 non-null   float64
 4   adaptation_members                     5278 non-null   float64
 5   has_adaptation_score                   5278 non-null   int64  
 6   has_adaptation_members                 5278 non-null   int64  
 7   has_prequel_score                      5278 non-null   int64  
 8   has_prequel_members                    5278 non-null   int64  
 9   has_prequel_type                       5278 non-null   int64  
 10  genre_Action                           5278 non-null   int64  
 11  genre_Adventure

In [239]:
other_scaler = StandardScaler()

features = ['adaptation_score', 'adaptation_members']

print(X_other_pre[features].isna().sum())

train_data = X_other_pre.iloc[train_idx][features]
val_data   = X_other_pre.iloc[val_idx][features]
test_data  = X_other_pre.iloc[test_idx][features]
real_data = real_other_pre[features]

imputer = SimpleImputer(strategy='mean') # or 'median'
train_imputed = imputer.fit_transform(train_data)
val_imputed   = imputer.transform(val_data)
test_imputed  = imputer.transform(test_data)
real_imputed = imputer.transform(real_data)

adaptation_train = other_scaler.fit_transform(train_imputed)
adaptation_val   = other_scaler.transform(val_imputed)
adaptation_test  = other_scaler.transform(test_imputed)
adaptation_real = other_scaler.transform(real_imputed)

print(adaptation_train)

adaptation_score      0
adaptation_members    0
dtype: int64
[[-0.52628175 -0.79872244]
 [-0.52628175 -0.79872244]
 [-0.52628175 -0.79872244]
 ...
 [-0.52628175 -0.79872244]
 [ 1.02495669  0.55516967]
 [-0.52628175 -0.79872244]]


In [240]:
X_other_pre.loc[train_idx, 'adaptation_score'] = adaptation_train[:, 0]
X_other_pre.loc[val_idx, 'adaptation_score'] = adaptation_val[:, 0]
X_other_pre.loc[test_idx, 'adaptation_score'] = adaptation_test[:, 0]
real_other_pre['adaptation_score'] = adaptation_real[:, 0]

X_other_pre.loc[train_idx, 'adaptation_members'] = adaptation_train[:, 1]
X_other_pre.loc[val_idx, 'adaptation_members'] = adaptation_val[:, 1]
X_other_pre.loc[test_idx, 'adaptation_members'] = adaptation_test[:, 1]
real_other_pre['adaptation_members'] = adaptation_real[:, 1]

print(X_other_pre.info())
print(real_other_pre.info())

<class 'pandas.DataFrame'>
RangeIndex: 5278 entries, 0 to 5277
Data columns (total 83 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   thumbnail                              5278 non-null   bool   
 1   prequel_score                          5278 non-null   float64
 2   prequel_members                        5278 non-null   float64
 3   adaptation_score                       5278 non-null   float64
 4   adaptation_members                     5278 non-null   float64
 5   has_adaptation_score                   5278 non-null   int64  
 6   has_adaptation_members                 5278 non-null   int64  
 7   has_prequel_score                      5278 non-null   int64  
 8   has_prequel_members                    5278 non-null   int64  
 9   has_prequel_type                       5278 non-null   int64  
 10  genre_Action                           5278 non-null   int64  
 11  genre_Adventure

In [241]:
class FusionNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        # Text projector
        self.text_projector = LearnedProjector(
            in_dim=768,
            hidden_dim=128,
            out_dim=64,
            dropout=0.4
        )

        # Image branch
        self.image_branch = nn.Sequential(
            nn.Linear(418, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

        # Tabular branch
        self.other_branch = nn.Sequential(
            nn.Linear(83, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

        self.studio_embedding = nn.EmbeddingBag(
            num_embeddings=n_studios,
            embedding_dim=8,
            mode='mean'
        )

        self.producer_embedding = nn.EmbeddingBag(
            num_embeddings=n_producers,
            embedding_dim=8,
            mode='mean'
        )

        # Fusion
        self.fusion = nn.Sequential(
            nn.Linear(64 + 64 + 32 + 8 + 8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def forward(self, text, image, other, studio_indices, studio_offsets, producer_indices, producer_offsets):
        # 1. Check raw inputs
        for name, tensor in [('text', text), ('image', image), ('other', other)]:
            if torch.isnan(tensor).any():
                print(f"NaN detected in raw input: {name}")

        text_features = self.text_projector(text)
        image_features = self.image_branch(image)
        other_features = self.other_branch(other)
        
        # 2. Check branches
        if torch.isnan(text_features).any(): print("NaN in text branch")
        if torch.isnan(image_features).any(): print("NaN in image branch (Check BatchNorm/Batch Size)")
        if torch.isnan(other_features).any(): print("NaN in other branch")

        studio_features = self.studio_embedding(studio_indices, studio_offsets)
        producer_features = self.producer_embedding(producer_indices, producer_offsets)
        
        # 3. Check embeddings
        if torch.isnan(studio_features).any(): print("NaN in studio embeddings")

        combined = torch.cat([text_features, image_features, other_features, studio_features, producer_features], dim=1)
        
        out = self.fusion(combined)
        if torch.isnan(out).any(): print("NaN generated inside Fusion layers")

        mean = out[:, 0]      # shape [32], not [32, 0:1]
        raw_std = out[:, 1]   # shape [32]

        std = torch.exp(raw_std) + 1e-6
        norm_dist = torch.distributions.Normal(mean, std)
        
        return norm_dist

In [242]:
X_text_train = torch.from_numpy(
    semantic_embeddings_train.astype(np.float32)
)

X_text_val = torch.from_numpy(
    semantic_embeddings_val.astype(np.float32)
)

X_text_test = torch.from_numpy(
    semantic_embeddings_test.astype(np.float32)
)

X_image_train = torch.from_numpy(
    image_train.astype(np.float32)
)

X_image_val = torch.from_numpy(
    image_val.astype(np.float32)
)

X_image_test = torch.from_numpy(
    image_test.astype(np.float32)
)

X_other = torch.from_numpy(
    X_other_pre.to_numpy(dtype=np.float32)
)

y_score = torch.tensor(
    df["score_z"].to_numpy(dtype=np.float32)
)

real_test = torch.from_numpy(
    real_semantic_embeddings.astype(np.float32)
)

real_image = torch.from_numpy(
    image_real.astype(np.float32)
)

real_other = torch.tensor(
    real_other_pre.to_numpy(dtype=np.float32)
)

In [243]:
other_train = X_other[train_idx]
other_val = X_other[val_idx]
other_test = X_other[test_idx]

score_train = y_score[train_idx]
score_val = y_score[val_idx]
score_test = y_score[test_idx]

In [244]:
model = FusionNetwork()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)


class AnimeDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        text,
        image,
        other,
        target,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets
    ):
        self.text = text
        self.image = image
        self.other = other
        self.target = target

        self.studio_indices = studio_indices
        self.studio_offsets = studio_offsets

        self.producer_indices = producer_indices
        self.producer_offsets = producer_offsets

    def __len__(self):
        return len(self.target)

    def __getitem__(self, idx):

        # Figure out where this anime's studio list starts
        studio_start = self.studio_offsets[idx]

        if idx + 1 < len(self.studio_offsets):
            studio_end = self.studio_offsets[idx + 1]
        else:
            studio_end = len(self.studio_indices)

        studio_indices = self.studio_indices[
            studio_start:studio_end
        ]

        # Same thing for producers
        producer_start = self.producer_offsets[idx]

        if idx + 1 < len(self.producer_offsets):
            producer_end = self.producer_offsets[idx + 1]
        else:
            producer_end = len(self.producer_indices)

        producer_indices = self.producer_indices[
            producer_start:producer_end
        ]

        return (
            self.text[idx],
            self.image[idx],
            self.other[idx],
            studio_indices,
            producer_indices,
            self.target[idx]
        )

def collate_fn(batch):

    texts = torch.stack([item[0] for item in batch])
    images = torch.stack([item[1] for item in batch])
    others = torch.stack([item[2] for item in batch])
    targets = torch.stack([item[5] for item in batch])

    studio_indices = []
    studio_offsets = []

    current_offset = 0

    for item in batch:
        indices = item[3]

        studio_offsets.append(current_offset)

        studio_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    studio_indices = torch.tensor(
        studio_indices,
        dtype=torch.long
    )

    studio_offsets = torch.tensor(
        studio_offsets,
        dtype=torch.long
    )

    producer_indices = []
    producer_offsets = []

    current_offset = 0

    for item in batch:
        indices = item[4]

        producer_offsets.append(current_offset)

        producer_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    producer_indices = torch.tensor(
        producer_indices,
        dtype=torch.long
    )

    producer_offsets = torch.tensor(
        producer_offsets,
        dtype=torch.long
    )

    return (
        texts,
        images,
        others,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets,
        targets
    )

In [245]:
train_dataset = AnimeDataset(
    X_text_train,
    X_image_train,
    other_train,
    score_train,
    studio_indices_train,
    studio_offsets_train,
    producer_indices_train,
    producer_offsets_train
)

val_dataset = AnimeDataset(
    X_text_val,
    X_image_val,
    other_val,
    score_val,
    studio_indices_val,
    studio_offsets_val,
    producer_indices_val,
    producer_offsets_val
)

test_dataset = AnimeDataset(
    X_text_test,
    X_image_test,
    other_test,
    score_test,
    studio_indices_test,
    studio_offsets_test,
    producer_indices_test,
    producer_offsets_test
)

In [246]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)


In [249]:
def nll_loss(dist, target):
    # print(dist.log_prob(target).shape)
    return -dist.log_prob(target).mean()

In [250]:
full_dataset = ConcatDataset([train_dataset, val_dataset])
n_samples = len(full_dataset)

k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

batch_size = train_loader.batch_size
collate_fn = train_loader.collate_fn 
patience = 15
n_epochs = 100

fold_results = []          
fold_model_states = []    

for fold, (train_idx, val_idx) in enumerate(kfold.split(np.arange(n_samples))):

    print(f"\n===== Fold {fold + 1}/{k} =====")

    model = FusionNetwork().to(device)  
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-4,
        weight_decay=1e-2
    )

    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)

    fold_train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn
    )
    fold_val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state = None

    for epoch in range(n_epochs):

        model.train()
        train_loss = 0

        for (
            text, image, other,
            studio_indices, studio_offsets,
            producer_indices, producer_offsets,
            target
        ) in fold_train_loader:

            text = text.to(device)
            image = image.to(device)
            other = other.to(device)
            studio_indices = studio_indices.to(device)
            studio_offsets = studio_offsets.to(device)
            producer_indices = producer_indices.to(device)
            producer_offsets = producer_offsets.to(device)
            target = target.to(device)

            prediction = model(
                text, image, other,
                studio_indices, studio_offsets,
                producer_indices, producer_offsets
            )

            loss = nll_loss(prediction, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(fold_train_loader)

        model.eval()
        val_loss = 0

        with torch.no_grad():
            for (
                text, image, other,
                studio_indices, studio_offsets,
                producer_indices, producer_offsets,
                target
            ) in fold_val_loader:

                text = text.to(device)
                image = image.to(device)
                other = other.to(device)
                studio_indices = studio_indices.to(device)
                studio_offsets = studio_offsets.to(device)
                producer_indices = producer_indices.to(device)
                producer_offsets = producer_offsets.to(device)
                target = target.to(device)

                prediction = model(
                    text, image, other,
                    studio_indices, studio_offsets,
                    producer_indices, producer_offsets
                )

                loss = nll_loss(prediction, target)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(fold_val_loader)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Fold {fold + 1} early stopping at epoch {epoch}.")
                break

        if epoch % 5 == 0 or patience_counter == 0:
            print(
                f"  Epoch {epoch}: "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f}"
            )

    print(f"Fold {fold + 1} best val loss: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)
    fold_model_states.append(best_model_state)

fold_results = np.array(fold_results)
print(f"\n===== CV Results ({k}-fold) =====")
print(f"Per-fold val loss: {fold_results}")
print(f"Mean: {fold_results.mean():.4f}  |  Std: {fold_results.std():.4f}")

best_fold_idx = fold_results.argmin()
best_model_state = fold_model_states[best_fold_idx]
model = FusionNetwork().to(device)
model.load_state_dict(best_model_state)
print(f"\nLoaded weights from fold {best_fold_idx + 1} (val loss {fold_results[best_fold_idx]:.4f})")


===== Fold 1/5 =====
  Epoch 0: Train Loss: 1.3529 | Val Loss: 1.3390
  Epoch 1: Train Loss: 1.2808 | Val Loss: 1.3083
  Epoch 2: Train Loss: 1.2244 | Val Loss: 1.2989
  Epoch 5: Train Loss: 1.0842 | Val Loss: 1.3196
  Epoch 10: Train Loss: 0.8589 | Val Loss: 1.6001
  Epoch 15: Train Loss: 0.7283 | Val Loss: 1.7104
Fold 1 early stopping at epoch 17.
Fold 1 best val loss: 1.2989

===== Fold 2/5 =====
  Epoch 0: Train Loss: 1.3611 | Val Loss: 1.2908
  Epoch 1: Train Loss: 1.2681 | Val Loss: 1.2607
  Epoch 2: Train Loss: 1.2193 | Val Loss: 1.2549
  Epoch 5: Train Loss: 1.0560 | Val Loss: 1.2719
  Epoch 10: Train Loss: 0.8480 | Val Loss: 1.4760
  Epoch 15: Train Loss: 0.7239 | Val Loss: 1.4807
Fold 2 early stopping at epoch 17.
Fold 2 best val loss: 1.2549

===== Fold 3/5 =====
  Epoch 0: Train Loss: 1.3651 | Val Loss: 1.2830
  Epoch 1: Train Loss: 1.2778 | Val Loss: 1.2795
  Epoch 2: Train Loss: 1.2361 | Val Loss: 1.2331
  Epoch 5: Train Loss: 1.0772 | Val Loss: 1.2782
  Epoch 10: Train 

In [255]:
model.eval()

total_nll = 0
total_samples = 0

all_predictions = []
all_targets = []

with torch.no_grad():

    for (
        text,
        image,
        other,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets,
        target
    ) in test_loader:

        text = text.to(device)
        image = image.to(device)
        other = other.to(device)

        studio_indices = studio_indices.to(device)
        studio_offsets = studio_offsets.to(device)

        producer_indices = producer_indices.to(device)
        producer_offsets = producer_offsets.to(device)

        target = target.to(device)

        predictions = model(
            text,
            image,
            other,
            studio_indices,
            studio_offsets,
            producer_indices,
            producer_offsets
        )

        nll = nll_loss(predictions, target)

        total_nll += nll * 32
        total_samples += target.size(0)



final_nll = (
    total_nll /
    total_samples
)


print(f"Test NLL:  {final_nll:.4f}")

Test NLL:  1.3051


### Current Anime Prediction

In [ ]:
# data prep

In [32]:
class AnimeInferenceDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        text,
        image,
        other,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets
    ):
        self.text = text
        self.image = image
        self.other = other

        self.studio_indices = studio_indices
        self.studio_offsets = studio_offsets

        self.producer_indices = producer_indices
        self.producer_offsets = producer_offsets

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):

        studio_start = self.studio_offsets[idx]

        if idx + 1 < len(self.studio_offsets):
            studio_end = self.studio_offsets[idx + 1]
        else:
            studio_end = len(self.studio_indices)

        studio_indices = self.studio_indices[
            studio_start:studio_end
        ]

        producer_start = self.producer_offsets[idx]

        if idx + 1 < len(self.producer_offsets):
            producer_end = self.producer_offsets[idx + 1]
        else:
            producer_end = len(self.producer_indices)

        producer_indices = self.producer_indices[
            producer_start:producer_end
        ]

        return (
            self.text[idx],
            self.image[idx],
            self.other[idx],
            studio_indices,
            producer_indices
        )

def inference_collate_fn(batch):

    texts = torch.stack(
        [item[0] for item in batch]
    )

    images = torch.stack(
        [item[1] for item in batch]
    )

    others = torch.stack(
        [item[2] for item in batch]
    )

    # -------------------------
    # Studios
    # -------------------------

    studio_indices = []
    studio_offsets = []

    current_offset = 0

    for item in batch:

        indices = item[3]

        studio_offsets.append(current_offset)

        studio_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    studio_indices = torch.tensor(
        studio_indices,
        dtype=torch.long
    )

    studio_offsets = torch.tensor(
        studio_offsets,
        dtype=torch.long
    )

    # -------------------------
    # Producers
    # -------------------------

    producer_indices = []
    producer_offsets = []

    current_offset = 0

    for item in batch:

        indices = item[4]

        producer_offsets.append(current_offset)

        producer_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    producer_indices = torch.tensor(
        producer_indices,
        dtype=torch.long
    )

    producer_offsets = torch.tensor(
        producer_offsets,
        dtype=torch.long
    )

    return (
        texts,
        images,
        others,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets
    )